# Physics CLT Intervention Demo

**Hypothesis:** Cross-layer transcoder (CLT) features encode physics-specific concepts (force, acceleration, energy, etc.) and causally influence the model's next-token predictions for physics completions.

**Protocol:**
1. Collect activations on physics prompts vs. matched non-physics controls.
2. Mine candidate features from activation deltas.
3. Ablate candidates on physics prompts (expect target logit to drop).
4. Insert/amplify candidates on control prompts (expect physics-related logit to rise).
5. Run open-ended generation with persistent interventions.

**Primary metric:** delta of target-token logit probability.  
**Secondary metrics:** rank shift of target token, qualitative generation drift.

## 1. Environment + Imports

In [ ]:
import gc
from collections import namedtuple
from functools import partial

import torch

from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils.demo_utils import (
    display_topk_token_predictions,
    display_generations_comparison,
    get_topk,
)
from circuit_tracer.utils import create_graph_files

Feature = namedtuple("Feature", ["layer", "pos", "feature_idx"])


def cleanup_vram():
    """Force-free GPU memory between heavy sections."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Load CLT-backed ReplacementModel

In [ ]:
MODEL_NAME = "google/gemma-2-2b"
TRANSCODER_SET = "mntss/clt-gemma-2-2b-426k"
BACKEND = "transformerlens"
DTYPE = torch.bfloat16

model = ReplacementModel.from_pretrained(
    MODEL_NAME, TRANSCODER_SET, dtype=DTYPE, backend=BACKEND
)

show_topk = partial(display_topk_token_predictions, tokenizer=model.tokenizer)

print(f"Model: {MODEL_NAME}")
print(f"CLT: {TRANSCODER_SET}")
print(f"Backend: {BACKEND}")
print(f"Layers: {model.cfg.n_layers}")
print(f"d_model: {model.cfg.d_model}")

## 3. Define Prompt Sets

Each physics prompt is paired with a syntactically similar control prompt from a different domain.  
The expected completion token for each physics prompt is listed alongside.

In [ ]:
prompt_pairs = [
    {
        "physics": "In physics, Newton's second law states that force equals mass times",
        "control": "In cooking, the golden rule states that flavor equals spice times",
        "target_token": "acceleration",
    },
    {
        "physics": "The unit of electrical resistance is the",
        "control": "The unit of monetary exchange is the",
        "target_token": "ohm",
    },
    {
        "physics": "Kinetic energy is equal to one half times mass times velocity",
        "control": "Total profit is equal to one half times revenue times margin",
        "target_token": "squared",
    },
    {
        "physics": "The speed of light in a vacuum is approximately",
        "control": "The speed of delivery in a city is approximately",
        "target_token": "3",
    },
    {
        "physics": "An object in free fall accelerates at 9.8 meters per second",
        "control": "A runner in the race accelerates at 2.5 meters per second",
        "target_token": "squared",
    },
]

print(f"Defined {len(prompt_pairs)} physics/control prompt pairs.")
for i, pair in enumerate(prompt_pairs):
    print(f"  [{i}] target='{pair['target_token']}' | {pair['physics'][:60]}...")

## 4. Baseline Activations + Logits

Collect activations and logits for every prompt.  
Shape of activations: `(n_layers, n_positions, d_transcoder)`.

In [ ]:
baselines = []

with torch.inference_mode():
    for pair in prompt_pairs:
        logits_phys, acts_phys = model.get_activations(pair["physics"])
        logits_ctrl, acts_ctrl = model.get_activations(pair["control"])
        baselines.append({
            "logits_phys": logits_phys,
            "acts_phys": acts_phys,
            "logits_ctrl": logits_ctrl,
            "acts_ctrl": acts_ctrl,
        })

print(f"Collected baselines for {len(baselines)} pairs.")
print(f"Activation tensor shape: {baselines[0]['acts_phys'].shape}")
print()

for i, (pair, bl) in enumerate(zip(prompt_pairs, baselines)):
    top = get_topk(bl["logits_phys"], model.tokenizer, k=5)
    top_str = ", ".join(f"{tok!r} ({p:.2%})" for tok, p in top)
    print(f"[{i}] Physics top-5: {top_str}")

## 5. Candidate Feature Mining

For each prompt pair, compute the activation delta at the **final token position** across all layers and features.  
Features that activate much more strongly on physics prompts than controls are candidates for physics-specific features.

In [ ]:
TOP_K_CANDIDATES = 8

all_candidates = []

for i, (pair, bl) in enumerate(zip(prompt_pairs, baselines)):
    acts_p = bl["acts_phys"]
    acts_c = bl["acts_ctrl"]

    min_pos = min(acts_p.shape[1], acts_c.shape[1])
    delta = acts_p[:, min_pos - 1, :] - acts_c[:, min_pos - 1, :]

    d_feat = delta.shape[-1]
    flat_topk = torch.topk(delta.flatten(), k=TOP_K_CANDIDATES)

    candidates = []
    for idx, val in zip(flat_topk.indices.tolist(), flat_topk.values.tolist()):
        layer = idx // d_feat
        feat = idx % d_feat
        act_val = float(acts_p[layer, min_pos - 1, feat])
        candidates.append({
            "layer": layer,
            "feature_idx": feat,
            "delta": val,
            "physics_activation": act_val,
        })

    all_candidates.append(candidates)
    print(f"\n[{i}] '{pair['target_token']}' — top {TOP_K_CANDIDATES} delta features:")
    for c in candidates:
        print(f"    L{c['layer']:>2d} feat {c['feature_idx']:>6d}  "
              f"delta={c['delta']:+.3f}  act={c['physics_activation']:.3f}")

## 6. Build Intervention Tuples

We build two sets of intervention tuples per prompt pair:
- **Ablation:** zero out top candidate features on the physics prompt.
- **Insertion:** inject physics-level activations into the control prompt.

Position is the final token of each prompt (accounting for the BOS token the model prepends).

In [ ]:
N_INTERVENE = 4

ablation_tuples_list = []
insertion_tuples_list = []

for i, (pair, bl, cands) in enumerate(zip(prompt_pairs, baselines, all_candidates)):
    phys_pos = bl["acts_phys"].shape[1] - 1
    ctrl_pos = bl["acts_ctrl"].shape[1] - 1

    top_cands = cands[:N_INTERVENE]

    ablation = [
        (c["layer"], phys_pos, c["feature_idx"], 0.0)
        for c in top_cands
    ]

    insertion = [
        (c["layer"], ctrl_pos, c["feature_idx"], c["physics_activation"])
        for c in top_cands
    ]

    ablation_tuples_list.append(ablation)
    insertion_tuples_list.append(insertion)

print(f"Built intervention tuples for {len(prompt_pairs)} pairs, {N_INTERVENE} features each.")
print(f"\nExample ablation tuple: {ablation_tuples_list[0][0]}")
print(f"Example insertion tuple: {insertion_tuples_list[0][0]}")

## 7. Logit-Level Causal Tests

### 7a. Ablation on Physics Prompts

Zeroing out the candidate features on a physics prompt should **decrease** the probability of the expected physics completion.

In [ ]:
ablation_results = []

with torch.inference_mode():
    for i, (pair, abl_tuples) in enumerate(zip(prompt_pairs, ablation_tuples_list)):
        prompt = pair["physics"]

        original_logits, _ = model.feature_intervention(prompt, [])
        ablated_logits, ablated_acts = model.feature_intervention(
            prompt,
            abl_tuples,
            constrained_layers=range(model.cfg.n_layers),
        )

        ablation_results.append({
            "original_logits": original_logits,
            "ablated_logits": ablated_logits,
            "ablated_acts": ablated_acts,
        })

        print(f"\n--- [{i}] ABLATION on: {prompt[:60]}... ---")
        print(f"    Target token: '{pair['target_token']}'")
        show_topk(prompt, original_logits, ablated_logits)

### 7b. Insertion on Control Prompts

Injecting physics-level activations into a control prompt should **increase** physics-related token probabilities.

In [ ]:
insertion_results = []

with torch.inference_mode():
    for i, (pair, ins_tuples) in enumerate(zip(prompt_pairs, insertion_tuples_list)):
        prompt = pair["control"]

        original_logits, _ = model.feature_intervention(prompt, [])
        inserted_logits, inserted_acts = model.feature_intervention(
            prompt,
            ins_tuples,
            constrained_layers=range(model.cfg.n_layers),
        )

        insertion_results.append({
            "original_logits": original_logits,
            "inserted_logits": inserted_logits,
            "inserted_acts": inserted_acts,
        })

        print(f"\n--- [{i}] INSERTION on: {prompt[:60]}... ---")
        print(f"    Target physics token: '{pair['target_token']}'")
        show_topk(prompt, original_logits, inserted_logits)

## 8. Activation-Level Effect Checks

Verify that the intervention actually changed activations at the targeted coordinates, and assess collateral activation changes at other features.

In [ ]:
for i, (pair, bl, cands, abl_res) in enumerate(
    zip(prompt_pairs, baselines, all_candidates, ablation_results)
):
    acts_orig = bl["acts_phys"]
    acts_abl = abl_res["ablated_acts"]
    if acts_abl is None:
        print(f"[{i}] No activation cache returned — skipping.")
        continue

    pos = acts_orig.shape[1] - 1
    print(f"\n[{i}] '{pair['target_token']}' — activation changes at final position:")

    for c in cands[:N_INTERVENE]:
        layer, feat = c["layer"], c["feature_idx"]
        orig_val = float(acts_orig[layer, pos, feat])
        new_val = float(acts_abl[layer, pos, feat])
        print(f"    L{layer:>2d} feat {feat:>6d}: {orig_val:.4f} -> {new_val:.4f} (delta={new_val - orig_val:+.4f})")

    total_delta = (acts_abl[:, pos, :] - acts_orig[:, pos, :]).abs().sum().item()
    targeted_delta = sum(
        abs(float(acts_abl[c["layer"], pos, c["feature_idx"]]) - float(acts_orig[c["layer"], pos, c["feature_idx"]]))
        for c in cands[:N_INTERVENE]
    )
    collateral_pct = 100 * (1.0 - targeted_delta / max(total_delta, 1e-12))
    print(f"    Collateral activation change: {collateral_pct:.1f}% of total delta is off-target")

## 9. Open-Ended Intervention Test

Persist the ablation across generated tokens using an open-ended slice and compare pre/post continuations.

In [ ]:
GEN_IDX = 0
MAX_NEW_TOKENS = 30

pair = prompt_pairs[GEN_IDX]
cands = all_candidates[GEN_IDX]
phys_prompt = pair["physics"]

seq_len = len(model.tokenizer(phys_prompt).input_ids)
last_pos = seq_len - 1
open_slice = slice(last_pos, None, None)

open_ablation_tuples = [
    (c["layer"], open_slice, c["feature_idx"], 0.0)
    for c in cands[:N_INTERVENE]
]

print(f"Prompt: {phys_prompt}")
print(f"Ablating {N_INTERVENE} features from position {last_pos} onward...")
print(f"Generating {MAX_NEW_TOKENS} tokens...\n")

pre_gen = [model.feature_intervention_generate(
    phys_prompt, [], do_sample=False, max_new_tokens=MAX_NEW_TOKENS, verbose=False
)[0]]

post_gen = [model.feature_intervention_generate(
    phys_prompt, open_ablation_tuples, do_sample=False, max_new_tokens=MAX_NEW_TOKENS, verbose=False
)[0]]

display_generations_comparison(phys_prompt, pre_gen, post_gen)

### Insertion generation test

Inject physics-level activations into the control prompt and observe generation drift toward physics content.

In [ ]:
ctrl_prompt = pair["control"]

ctrl_seq_len = len(model.tokenizer(ctrl_prompt).input_ids)
ctrl_last_pos = ctrl_seq_len - 1
ctrl_open_slice = slice(ctrl_last_pos, None, None)

open_insertion_tuples = [
    (c["layer"], ctrl_open_slice, c["feature_idx"], c["physics_activation"])
    for c in cands[:N_INTERVENE]
]

print(f"Control prompt: {ctrl_prompt}")
print(f"Inserting {N_INTERVENE} physics features from position {ctrl_last_pos} onward...\n")

pre_ctrl_gen = [model.feature_intervention_generate(
    ctrl_prompt, [], do_sample=False, max_new_tokens=MAX_NEW_TOKENS, verbose=False
)[0]]

post_ctrl_gen = [model.feature_intervention_generate(
    ctrl_prompt, open_insertion_tuples, do_sample=False, max_new_tokens=MAX_NEW_TOKENS, verbose=False
)[0]]

display_generations_comparison(ctrl_prompt, pre_ctrl_gen, post_ctrl_gen)

## 10. Attribution Graph Cross-Check (Optional)

Run attribution on one representative physics prompt and verify that mined candidate features appear among the most influential nodes in the attribution graph.

In [ ]:
ATTR_IDX = 0

attr_prompt = prompt_pairs[ATTR_IDX]["physics"]
print(f"Running attribution on: {attr_prompt}")

graph = attribute(
    prompt=attr_prompt,
    model=model,
    max_n_logits=10,
    desired_logit_prob=0.95,
    batch_size=256,
    max_feature_nodes=4096,
    offload="cpu",
    verbose=True,
)

In [ ]:
active_feats = graph.active_features[graph.selected_features]
selected_set = set()
for row in active_feats:
    layer, pos, feat_idx = row.tolist()
    selected_set.add((layer, feat_idx))

cands = all_candidates[ATTR_IDX]

print(f"Attribution graph has {len(graph.selected_features)} selected feature nodes.")
print(f"\nCandidate features vs. attribution graph:")
for c in cands[:TOP_K_CANDIDATES]:
    present = (c["layer"], c["feature_idx"]) in selected_set
    marker = "IN GRAPH" if present else "not in graph"
    print(f"    L{c['layer']:>2d} feat {c['feature_idx']:>6d}  delta={c['delta']:+.3f}  [{marker}]")

## 11. Results Summary Table

Aggregate per-prompt metrics into a compact table for quick evaluation.

In [ ]:
from IPython.display import HTML


def find_token_rank_and_prob(logits, tokenizer, target_token):
    """Find the rank and probability of a target token in logit output."""
    probs = torch.softmax(logits.squeeze()[-1], dim=-1)
    sorted_probs, sorted_indices = probs.sort(descending=True)

    target_ids = tokenizer.encode(target_token, add_special_tokens=False)
    if not target_ids:
        return None, None

    target_id = target_ids[0]
    rank_mask = (sorted_indices == target_id).nonzero(as_tuple=True)[0]
    if len(rank_mask) == 0:
        return None, None

    rank = rank_mask[0].item()
    prob = sorted_probs[rank].item()
    return rank, prob


rows_html = ""
n_pass = 0

for i, (pair, abl_res) in enumerate(zip(prompt_pairs, ablation_results)):
    target = pair["target_token"]

    rank_orig, prob_orig = find_token_rank_and_prob(
        abl_res["original_logits"], model.tokenizer, target
    )
    rank_abl, prob_abl = find_token_rank_and_prob(
        abl_res["ablated_logits"], model.tokenizer, target
    )

    if rank_orig is None or rank_abl is None:
        rows_html += f"<tr><td>{i}</td><td>{target}</td><td colspan='5'>token not found in vocab</td></tr>"
        continue

    delta_prob = prob_abl - prob_orig
    rank_shift = rank_abl - rank_orig
    passed = delta_prob < 0
    if passed:
        n_pass += 1
    color = "#27AE60" if passed else "#E74C3C"
    verdict = "PASS" if passed else "FAIL"

    rows_html += (
        f"<tr>"
        f"<td>{i}</td>"
        f"<td><code>{target}</code></td>"
        f"<td>{prob_orig:.4f}</td>"
        f"<td>{prob_abl:.4f}</td>"
        f"<td>{delta_prob:+.4f}</td>"
        f"<td>{rank_orig} &rarr; {rank_abl} ({rank_shift:+d})</td>"
        f"<td style='color:{color};font-weight:bold'>{verdict}</td>"
        f"</tr>"
    )

table_html = f"""
<h3>Ablation Results: does zeroing candidate features decrease target-token probability?</h3>
<table border='1' cellpadding='5' cellspacing='0' style='border-collapse:collapse;font-family:monospace;font-size:13px'>
<tr style='background:#eee'>
  <th>#</th><th>Target</th><th>P(orig)</th><th>P(ablated)</th>
  <th>&Delta;P</th><th>Rank shift</th><th>Pass?</th>
</tr>
{rows_html}
</table>
<p><b>{n_pass}/{len(prompt_pairs)}</b> prompts show expected decrease in target-token probability after ablation.</p>
"""

display(HTML(table_html))

## 12. VRAM Cleanup

Run this cell to free GPU memory before restarting experiments or loading a different model.  
For a full reset, **restart the kernel** (Kernel > Restart) which guarantees all GPU memory is released.

In [ ]:
for name in list(globals()):
    obj = globals()[name]
    if isinstance(obj, torch.Tensor) or hasattr(obj, "parameters"):
        if name not in ("torch",):
            del globals()[name]

cleanup_vram()

if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print(f"GPU memory — allocated: {alloc:.2f} GB, reserved: {reserved:.2f} GB")
else:
    print("No CUDA device.")

print("\nFor a full VRAM reset, restart the kernel: Kernel > Restart.")